In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ── Copy and extract your waste dataset ZIP from Google Drive ─────────────────
# Update the filename below to match your actual ZIP file name.
!cp "/content/drive/MyDrive/DATASET FOR WISE.zip" "/content/"
!unzip -q "/content/DATASET FOR WISE.zip" -d "/content"

import os
count = 0
for root, dirs, files in os.walk('/content/DATASET FOR WISE'):
    count += len(files)

print("Upload + Extraction Complete!")
print("Total files:", count)


## Part 1 — Setup

Imports, GPU check, reproducibility, global config, and the shared plot style.


In [ ]:

import os
import json
import shutil
import random
import zipfile
from datetime import datetime
from collections import Counter

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, FancyBboxPatch
import seaborn as sns

from sklearn.metrics import classification_report, confusion_matrix, f1_score

print(f"TensorFlow version: {tf.__version__}")
print(f"Keras version: {keras.__version__}")

# --- GPU detection ---
gpu_devices = tf.config.list_physical_devices("GPU")
if gpu_devices:
    print(f"GPU detected: {gpu_devices[0].name}")
    try:
        for gpu in gpu_devices:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(f"Could not set memory growth: {e}")
else:
    print("No GPU detected. Training will run on CPU and will be slow.")
    print("In Colab: Runtime > Change runtime type > T4 GPU")

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# --- Global configuration ---
# ── UPDATE THIS to the path of your waste dataset folder ────────────────────
DATASET_DIR = "/content/DATASET FOR WISE"
OUTPUT_DIR = "/content/waste_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMG_SIZE = (224, 224)
IMG_SHAPE = IMG_SIZE + (3,)
BATCH_SIZE = 32

TRAIN_SPLIT = 0.70
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15


EPOCHS_PHASE1 = 25
EPOCHS_PHASE2 = 30          # was 15 — more room for fine-tuning to converge
FINE_TUNE_LAYERS = 80       # unfreezes top EfficientNetB0 blocks (BatchNorm layers stay frozen)
DENSE_UNITS = 256           # was 128 — more head capacity for 22 similar classes
DROPOUT_RATE = 0.4          # was 0.3 — extra regularization for the deeper fine-tune
TARGET_ACCURACY = 0.90
USE_SMART_CROP = False      # optional vegetation-bounding-box crop, see Part 2


COLORS = {
    "accuracy":  "#16A34A",   # emerald — primary brand green
    "loss":      "#DC2626",   # red
    "precision": "#2563EB",   # blue
    "recall":    "#D97706",   # amber
    "correct":   "#16A34A",
    "wrong":     "#DC2626",
    "text":      "#0F172A",   # near-black, titles and body text
    "subtext":   "#64748B",   # muted gray, captions and axis labels
    "grid":      "#E2E8F0",   # light gridlines
    "panel":     "#F1F5F9",   # soft background panels and cards
    "border":    "#CBD5E1",   # thin borders on cards and image tiles
}

def setup_plot_style():
    """Apply one consistent, clean, presentation-ready style to every figure."""
    style = {
        "figure.dpi": 100,
        "savefig.dpi": 300,
        "figure.facecolor": "white",
        "savefig.facecolor": "white",
        "font.size": 12,
        "text.color": COLORS["text"],
        "axes.titlesize": 15,
        "axes.titleweight": "bold",
        "axes.titlepad": 14,
        "axes.labelsize": 11,
        "axes.labelcolor": COLORS["subtext"],
        "axes.facecolor": "white",
        "axes.edgecolor": COLORS["border"],
        "axes.linewidth": 0.8,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "axes.axisbelow": True,
        "grid.color": COLORS["grid"],
        "grid.alpha": 0.8,
        "grid.linewidth": 0.8,
        "grid.linestyle": "-",
        "xtick.labelsize": 10,
        "ytick.labelsize": 10,
        "xtick.color": COLORS["subtext"],
        "ytick.color": COLORS["subtext"],
        "xtick.major.size": 0,
        "ytick.major.size": 0,
        "legend.fontsize": 10,
        "legend.frameon": False,
    }
    # Only set keys this Matplotlib build recognizes, so an older Colab
    # runtime never crashes on a newer rcParam name.
    for key, value in style.items():
        if key in plt.rcParams:
            plt.rcParams[key] = value
    sns.set_style("white")

setup_plot_style()
print(f"\nOutputs will be saved to: {OUTPUT_DIR}")

## Part 2 — Data loading

Detects waste classes automatically from your dataset folder structure,
builds the 70/15/15 split, and computes class weights.


In [ ]:


if not os.path.isdir(DATASET_DIR):
    raise FileNotFoundError(
        f"Dataset directory not found: {DATASET_DIR}\n"
        "Update DATASET_DIR in Part 1 to point at your dataset folder."
    )

# --- Discover classes automatically ---
CLASS_NAMES = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
NUM_CLASSES = len(CLASS_NAMES)

if NUM_CLASSES == 0:
    raise ValueError(f"No class folders found inside {DATASET_DIR}")

print(f"Total classes detected: {NUM_CLASSES}")
print(f"Class names: {CLASS_NAMES}\n")

VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp")

# --- Build a stratified train/val/test split for every class ---
train_paths, train_labels = [], []
val_paths, val_labels = [], []
test_paths, test_labels = [], []

class_counts = {}

for class_idx, class_name in enumerate(CLASS_NAMES):
    class_dir = os.path.join(DATASET_DIR, class_name)
    files = [
        os.path.join(class_dir, f) for f in os.listdir(class_dir)
        if f.lower().endswith(VALID_EXT)
    ]
    rng = random.Random(SEED)
    rng.shuffle(files)

    n = len(files)
    class_counts[class_name] = n

    n_train = int(n * TRAIN_SPLIT)
    n_val = int(n * VAL_SPLIT)
    # remainder goes to test so rounding never drops an image

    train_files = files[:n_train]
    val_files = files[n_train:n_train + n_val]
    test_files = files[n_train + n_val:]

    train_paths.extend(train_files); train_labels.extend([class_idx] * len(train_files))
    val_paths.extend(val_files);     val_labels.extend([class_idx] * len(val_files))
    test_paths.extend(test_files);   test_labels.extend([class_idx] * len(test_files))

    print(f"{class_name:30s} total={n:5d}  train={len(train_files):4d}  val={len(val_files):4d}  test={len(test_files):4d}")

print(f"\nTotal images: {sum(class_counts.values())}")
print(f"Train images: {len(train_paths)}")
print(f"Validation images: {len(val_paths)}")
print(f"Test images: {len(test_paths)}")

# --- Class weights (inverse frequency) — upweights under-represented
#     classes in the training loss only, never in val/test ---
train_label_counts = Counter(train_labels)
class_weight_dict = {
    idx: len(train_labels) / (NUM_CLASSES * train_label_counts[idx])
    for idx in range(NUM_CLASSES)
}
train_weights = [class_weight_dict[l] for l in train_labels]

print("\nClass weights (1.0 = average representation):")
for idx, name in enumerate(CLASS_NAMES):
    print(f"  {name:30s} weight={class_weight_dict[idx]:.2f}")

# --- Optional smart crop: keep only the vegetation bounding box ---
def _smart_crop_numpy(img_uint8):
    img = img_uint8.astype(np.float32)
    r, g, b = img[..., 0], img[..., 1], img[..., 2]
    veg_mask = (g >= r * 0.85) & ((g + r) >= b * 1.2)
    ys, xs = np.where(veg_mask)
    h, w = veg_mask.shape
    if len(xs) < 0.05 * h * w:
        return img_uint8  # mask too small/unreliable, keep original
    x0, x1, y0, y1 = xs.min(), xs.max(), ys.min(), ys.max()
    mx, my = int((x1 - x0) * 0.08), int((y1 - y0) * 0.08)
    x0, x1 = max(0, x0 - mx), min(w, x1 + mx)
    y0, y1 = max(0, y0 - my), min(h, y1 + my)
    return img_uint8[y0:y1, x0:x1, :]

def smart_crop(image):
    cropped = tf.py_function(
        func=lambda im: _smart_crop_numpy(im.numpy().astype(np.uint8)),
        inp=[image],
        Tout=tf.uint8,
    )
    cropped.set_shape([None, None, 3])
    return cropped

# --- Build tf.data pipelines ---
def load_and_preprocess(path, label):
    image = tf.io.read_file(path)
    image = tf.image.decode_image(image, channels=3, expand_animations=False)
    if USE_SMART_CROP:
        image = smart_crop(image)
    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)
    image = preprocess_input(image)  # EfficientNetB0 expects float32 pixels in [0, 255] (built-in Rescaling/Normalization inside backbone)
    label = tf.one_hot(label, NUM_CLASSES)
    return image, label

def make_dataset(paths, labels, weights=None, shuffle=False, batch_size=BATCH_SIZE):
    if weights is not None:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))
    else:
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))

    if shuffle:
        ds = ds.shuffle(buffer_size=len(paths), seed=SEED)

    if weights is not None:
        def _load(path, label, weight):
            image, label = load_and_preprocess(path, label)
            return image, label, weight
        ds = ds.map(_load, num_parallel_calls=tf.data.AUTOTUNE)
    else:
        ds = ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE)

    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds_raw = make_dataset(train_paths, train_labels, weights=train_weights, shuffle=True)
val_ds = make_dataset(val_paths, val_labels, shuffle=False)
test_ds = make_dataset(test_paths, test_labels, shuffle=False)

print("\ntf.data pipelines created: train_ds_raw (weighted), val_ds, test_ds")

## Part 3 — Data augmentation

Flip, rotation, and zoom at the same strength as before; contrast/brightness toned down.

In [ ]:
# ── WEAK_CLASSES: list class names that are under-represented in your dataset.
# Leave this empty first; after your first training run check the per-class
# F1 scores in the classification report and add the low-performing classes here.
WEAK_CLASSES = []   # e.g. ["Glass", "Organic"] once you know which are weak

_missing = [c for c in WEAK_CLASSES if c not in CLASS_NAMES]
if _missing:
    print(f"WARNING: WEAK_CLASSES not found in CLASS_NAMES, skipping: {_missing}")
WEAK_CLASS_INDICES = [CLASS_NAMES.index(c) for c in WEAK_CLASSES if c in CLASS_NAMES]

# RandomBrightness needs TF >= 2.9 — guard so this still runs on older TF
_has_random_brightness = hasattr(layers, "RandomBrightness")

_PIXEL_VALUE_RANGE = (0.0, 255.0)  # EfficientNetB0 input pixel range [0, 255]

def _build_aug_stack(name, rotation, zoom, contrast, brightness):
    stack = [
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(rotation),
        layers.RandomZoom(zoom),
        layers.RandomContrast(contrast, value_range=_PIXEL_VALUE_RANGE),
    ]
    if _has_random_brightness:
        stack.append(layers.RandomBrightness(brightness, value_range=_PIXEL_VALUE_RANGE))
    else:
        print(f"NOTE: layers.RandomBrightness not available in this TF version "
              f"({tf.__version__}) - '{name}' will skip brightness jitter.")
    return keras.Sequential(stack, name=name)

base_augmentation   = _build_aug_stack(
    "base_augmentation",   rotation=0.12, zoom=0.12, contrast=0.10, brightness=0.10
)
strong_augmentation = _build_aug_stack(
    "strong_augmentation", rotation=0.25, zoom=0.25, contrast=0.20, brightness=0.20
)

_weak_mask_np = np.zeros(NUM_CLASSES, dtype=bool)
_weak_mask_np[WEAK_CLASS_INDICES] = True
weak_mask_lookup = tf.constant(_weak_mask_np)

def augment(image, label, weight):
    class_idx = tf.argmax(label, axis=-1)
    is_weak   = tf.gather(weak_mask_lookup, class_idx)

    base_out   = base_augmentation(image,   training=True)
    strong_out = strong_augmentation(image, training=True)

    mask  = tf.reshape(is_weak, [-1, 1, 1, 1])
    image = tf.where(mask, strong_out, base_out)
    return image, label, weight

train_ds = train_ds_raw.map(augment, num_parallel_calls=tf.data.AUTOTUNE)

print("Two-tier augmentation attached to train_ds only (val_ds/test_ds untouched):")
print(f"  base_augmentation   -> all {NUM_CLASSES} classes (mild)")
print(f"  strong_augmentation -> {len(WEAK_CLASS_INDICES)} weak classes (stronger):")
for i in WEAK_CLASS_INDICES:
    print(f"    - {CLASS_NAMES[i]}")


In [ ]:
from PIL import Image
import os

print("Checking dataset...")

fixed = 0
failed = 0

for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:

        if file.lower().endswith((".jpg",".jpeg",".png",".bmp",".webp")):

            path = os.path.join(root,file)

            try:
                img = Image.open(path)

                if img.mode != "RGB":
                    img = img.convert("RGB")
                    img.save(path)

                    fixed += 1

            except Exception as e:
                print("ERROR:", path)
                print(e)
                failed += 1

print(f"\nConverted to RGB: {fixed}")
print(f"Failed images: {failed}")
print("Dataset check complete.")

## Part 4 — Sample images per class

One representative image per class, saved as `sample_images_per_class.png`.


In [ ]:


def show_one_sample_per_class(save_path):
    cols = 5
    rows = int(np.ceil(NUM_CLASSES / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.7, rows * 3.1))
    fig.patch.set_facecolor("white")
    axes = np.array(axes).reshape(-1)

    for idx, class_name in enumerate(CLASS_NAMES):
        ax = axes[idx]
        class_dir = os.path.join(DATASET_DIR, class_name)
        candidates = [f for f in os.listdir(class_dir) if f.lower().endswith(VALID_EXT)]
        if not candidates:
            ax.axis("off")
            continue
        img = tf.io.read_file(os.path.join(class_dir, candidates[0]))
        img = tf.image.decode_image(img, channels=3, expand_animations=False)

        ax.imshow(img.numpy())
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(COLORS["border"])
            spine.set_linewidth(1.0)
        ax.set_xlabel(class_name.replace("_", " "), fontsize=8.5,
                      color=COLORS["text"], labelpad=6)

    for idx in range(NUM_CLASSES, len(axes)):
        axes[idx].axis("off")

    fig.suptitle(f"Sample image per class  \u00b7  {NUM_CLASSES} classes",
                 fontsize=16, fontweight="bold", color=COLORS["text"], y=1.01)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

show_one_sample_per_class(os.path.join(OUTPUT_DIR, "sample_images_per_class.png"))

## Part 5 — Model architecture

EfficientNetB0 backbone (frozen, ImageNet weights) with a wider classification head.

In [ ]:


base_model = EfficientNetB0(
    input_shape=IMG_SHAPE,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

inputs = keras.Input(shape=IMG_SHAPE)
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.BatchNormalization()(x)
x = layers.Dense(DENSE_UNITS, activation="relu")(x)
x = layers.Dropout(DROPOUT_RATE)(x)
outputs = layers.Dense(NUM_CLASSES, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="waste_classifier_efficientnetb0")
model.summary()

print(f"\nTotal EfficientNetB0 layers: {len(base_model.layers)}")

## Part 6 — Training phase 1

Trains the classification head with the backbone frozen.

In [ ]:


model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)

# Temporary training checkpoint (stored outside OUTPUT_DIR so only TFLite models are exported)
BEST_MODEL_PATH = "/content/best_model.keras"

callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1
    ),
]

# class_weight isn't passed to fit() directly — tf.data inputs from Part 2
# already carry a per-sample weight as the 3rd element of train_ds, which
# Keras applies automatically as (x, y, sample_weight).
print("Starting Phase 1 training (frozen EfficientNetB0 backbone)...\n")
history_phase1 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE1,
    callbacks=callbacks_phase1,
)

## Part 7 — Fine-tuning phase

Unfreezes the last 80 non-BatchNorm EfficientNetB0 layers and continues training at a lower learning rate.

In [ ]:


base_model.trainable = True

# Freeze everything except the last FINE_TUNE_LAYERS layers,
# and ALWAYS keep BatchNormalization layers frozen for EfficientNet fine-tuning
# to prevent moving-mean/variance drift and validation accuracy collapse.
for layer in base_model.layers[:-FINE_TUNE_LAYERS]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(
    f"Unfroze {trainable_count} of {len(base_model.layers)} EfficientNetB0 layers "
    "for fine-tuning (BatchNorm layers kept frozen)."
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.0001),
    loss=keras.losses.CategoricalCrossentropy(label_smoothing=0.05),
    metrics=[
        "accuracy",
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
    ],
)

callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=6, restore_best_weights=True, verbose=1
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1
    ),
    keras.callbacks.ModelCheckpoint(
        BEST_MODEL_PATH, monitor="val_loss", save_best_only=True, verbose=1
    ),
]

print("\nStarting Phase 2 training (fine-tuning)...\n")
history_phase2 = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS_PHASE2,
    callbacks=callbacks_phase2,
)

# --- Merge both training histories into one continuous record ---
def merge_histories(h1, h2):
    merged = {}
    for key in h1.history:
        merged[key] = h1.history[key] + h2.history.get(key, [])
    return merged

full_history = merge_histories(history_phase1, history_phase2)
phase1_epochs = len(history_phase1.history["loss"])
print(f"\nPhase 1 epochs: {phase1_epochs} | Phase 2 epochs: {len(history_phase2.history['loss'])}")

## Part 8 — Training curves

Accuracy, loss, precision, and recall over both training phases.

In [ ]:


def plot_metric(history, metric_key, val_key, color, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(8.5, 5))
    fig.patch.set_facecolor("white")

    epochs_range = list(range(1, len(history[metric_key]) + 1))
    train_vals = history[metric_key]
    val_vals = history[val_key]

    if phase1_epochs < len(epochs_range):
        ax.axvspan(phase1_epochs + 0.5, epochs_range[-1], color=COLORS["panel"], zorder=0)
        ax.text((phase1_epochs + epochs_range[-1] + 1) / 2, 1.015, "fine-tuning",
                transform=ax.get_xaxis_transform(), ha="center", va="bottom",
                fontsize=9, color=COLORS["subtext"])

    ax.plot(epochs_range, train_vals, color=color, linewidth=2.2,
            label="Training", zorder=3)
    ax.fill_between(epochs_range, train_vals, alpha=0.08, color=color, zorder=1)
    ax.plot(epochs_range, val_vals, color=color, linewidth=2.2, linestyle="--",
            alpha=0.55, label="Validation", zorder=3)

    ax.scatter(epochs_range[-1], train_vals[-1], color=color, s=32,
               zorder=5, edgecolor="white", linewidth=1)
    ax.scatter(epochs_range[-1], val_vals[-1], color=color, s=32, alpha=0.55,
               zorder=5, edgecolor="white", linewidth=1)

    ax.set_title(title, loc="left")
    ax.set_xlabel("Epoch")
    ax.set_ylabel(ylabel)
    ax.set_xlim(1, epochs_range[-1])
    ax.legend(loc="lower right" if metric_key != "loss" else "upper right",
              frameon=False, ncol=2, handlelength=1.6)

    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, filename)
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

plot_metric(full_history, "accuracy", "val_accuracy", COLORS["accuracy"],
            "Training Accuracy vs Epochs", "Accuracy", "training_accuracy.png")
plot_metric(full_history, "loss", "val_loss", COLORS["loss"],
            "Training Loss vs Epochs", "Loss", "training_loss.png")
plot_metric(full_history, "precision", "val_precision", COLORS["precision"],
            "Training Precision vs Epochs", "Precision", "training_precision.png")
plot_metric(full_history, "recall", "val_recall", COLORS["recall"],
            "Training Recall vs Epochs", "Recall", "training_recall.png")

## Part 9 — Evaluation on the test set

Test metrics, a check against the 90% target, classification report, and confusion matrix.

In [ ]:


test_results = model.evaluate(test_ds, return_dict=True, verbose=1)
test_loss = test_results["loss"]
test_accuracy = test_results["accuracy"]
test_precision = test_results["precision"]
test_recall = test_results["recall"]

# --- Collect predictions for the whole test set ---
y_true, y_pred, y_conf = [], [], []
for images, labels in test_ds:
    preds = model.predict(images, verbose=0)
    y_true.extend(np.argmax(labels.numpy(), axis=1))
    y_pred.extend(np.argmax(preds, axis=1))
    y_conf.extend(np.max(preds, axis=1))

y_true = np.array(y_true)
y_pred = np.array(y_pred)
y_conf = np.array(y_conf)

test_f1 = f1_score(y_true, y_pred, average="weighted")

print(f"\nTest Loss:      {test_loss:.4f}")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {test_precision:.4f}")
print(f"Test Recall:    {test_recall:.4f}")
print(f"Test F1 Score:  {test_f1:.4f}")

if test_accuracy >= TARGET_ACCURACY:
    print(f"\nTarget accuracy of {TARGET_ACCURACY*100:.0f}% reached.")
else:
    gap = (TARGET_ACCURACY - test_accuracy) * 100
    print(f"\nTarget accuracy of {TARGET_ACCURACY*100:.0f}% not yet reached "
          f"(currently {test_accuracy*100:.1f}%, {gap:.1f} points short).")

# --- Classification report (text) ---
report_text = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
report_path = os.path.join(OUTPUT_DIR, "classification_report.txt")
with open(report_path, "w") as f:
    f.write(report_text)
print(f"\nSaved: {report_path}")
print(report_text)

# --- Classification report (visual table) ---
report_dict = classification_report(y_true, y_pred, target_names=CLASS_NAMES,
                                     digits=4, output_dict=True)

def save_classification_report_visual(report_dict, class_names, save_path):
    rows = class_names + ["accuracy", "macro avg", "weighted avg"]
    cols = ["precision", "recall", "f1-score", "support"]
    table_data = []
    for r in rows:
        if r == "accuracy":
            table_data.append(["", "", f"{report_dict['accuracy']:.4f}", ""])
        else:
            d = report_dict[r]
            table_data.append([f"{d['precision']:.4f}", f"{d['recall']:.4f}",
                                f"{d['f1-score']:.4f}", f"{int(d['support'])}"])

    n_body = len(class_names)
    fig_height = 0.36 * len(rows) + 1.6
    fig, ax = plt.subplots(figsize=(8.5, fig_height))
    fig.patch.set_facecolor("white")
    ax.axis("off")

    table = ax.table(cellText=table_data,
                      rowLabels=[r.replace("_", " ") for r in rows],
                      colLabels=[c.title() for c in cols],
                      loc="center", cellLoc="center")
    table.auto_set_font_size(False)
    table.set_fontsize(9.5)
    table.scale(1, 1.5)

    for (row, col), cell in table.get_celld().items():
        cell.set_edgecolor(COLORS["grid"])
        cell.set_linewidth(0.6)
        if row == 0:
            cell.set_facecolor(COLORS["text"])
            cell.set_text_props(color="white", fontweight="bold")
        elif row > n_body:
            cell.set_facecolor(COLORS["panel"])
            cell.set_text_props(fontweight="bold", color=COLORS["text"])
        else:
            cell.set_facecolor("white" if row % 2 else "#FBFCFD")
            cell.set_text_props(color=COLORS["text"])

    ax.set_title("Classification report", loc="left", color=COLORS["text"], pad=18)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

save_classification_report_visual(report_dict, CLASS_NAMES,
                                   os.path.join(OUTPUT_DIR, "classification_report_visual.png"))

# --- Confusion matrix (high quality) ---
def save_confusion_matrix(y_true, y_pred, class_names, save_path):
    from matplotlib.colors import LinearSegmentedColormap
    cm = confusion_matrix(y_true, y_pred)
    n = len(class_names)
    cmap = LinearSegmentedColormap.from_list(
        "plant_green", ["#F0FDF4", "#4ADE80", "#15803D", "#14532D"]
    )

    fig, ax = plt.subplots(figsize=(max(12, n * 0.52), max(10, n * 0.52)))
    fig.patch.set_facecolor("white")
    sns.heatmap(cm, annot=True, fmt="d", cmap=cmap, cbar=True,
                xticklabels=class_names, yticklabels=class_names,
                linewidths=1.2, linecolor="white", ax=ax, square=True,
                annot_kws={"fontsize": max(7, 11 - n // 6)},
                cbar_kws={"shrink": 0.65, "label": "Predictions", "pad": 0.02})

    ax.set_title("Confusion matrix \u2014 test set", loc="left",
                 color=COLORS["text"], pad=16, fontsize=17)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
    plt.xticks(rotation=42, ha="right", fontsize=9.5)
    plt.yticks(rotation=0, fontsize=9.5)

    cbar = ax.collections[0].colorbar
    cbar.outline.set_visible(False)
    cbar.ax.tick_params(labelsize=9, color=COLORS["subtext"])

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

save_confusion_matrix(y_true, y_pred, CLASS_NAMES,
                       os.path.join(OUTPUT_DIR, "high_quality_confusion_matrix.png"))
shutil.copy(os.path.join(OUTPUT_DIR, "high_quality_confusion_matrix.png"),
            os.path.join(OUTPUT_DIR, "confusion_matrix.png"))

## Part 10 — Correct test predictions (per class)

For every class, only the test images the model got RIGHT, styled green throughout.

In [ ]:


CORRECT_SAMPLES_PER_CLASS = 4
correct_rng = np.random.RandomState(SEED)

def load_image_for_display(path):
    img = tf.io.read_file(path)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)
    return img.numpy().astype("uint8")

def save_correct_prediction_grid(class_name, class_idx, save_path,
                                  max_display=CORRECT_SAMPLES_PER_CLASS):
    candidates = [
        i for i in range(len(test_paths))
        if y_true[i] == class_idx and y_pred[i] == class_idx
    ]
    if not candidates:
        print(f"  {class_name:30s} -> 0 correct predictions, skipped")
        return

    n_sample = min(max_display, len(candidates))
    sample = list(correct_rng.choice(candidates, size=n_sample, replace=False))

    fig, axes = plt.subplots(1, len(sample), figsize=(len(sample) * 3.5, 4.6))
    fig.patch.set_facecolor("white")
    if len(sample) == 1:
        axes = [axes]

    for ax, idx in zip(axes, sample):
        img = load_image_for_display(test_paths[idx])
        confidence = y_conf[idx]

        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(COLORS["correct"])
            spine.set_linewidth(3)

        track_ax = ax.inset_axes([0.0, -0.14, 1.0, 0.05])
        track_ax.barh([0], [1], color=COLORS["panel"], height=1)
        track_ax.barh([0], [confidence], color=COLORS["correct"], height=1)
        track_ax.set_xlim(0, 1)
        track_ax.axis("off")

        ax.set_xlabel(f"{confidence*100:.1f}% confidence", fontsize=9.5,
                      color=COLORS["correct"], fontweight="bold", labelpad=16)

    fig.suptitle(f"Correct predictions \u2014 {class_name.replace('_', ' ')}",
                 fontsize=14, fontweight="bold", color=COLORS["text"], y=1.04)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

correct_total = int((y_true == y_pred).sum())
print(f"Total correct test predictions: {correct_total} / {len(y_true)}\n")
print("Generating CORRECT per-class prediction grids...\n")
for class_idx, class_name in enumerate(CLASS_NAMES):
    filename = f"correct_predictions_{class_name}.png"
    save_correct_prediction_grid(class_name, class_idx, os.path.join(OUTPUT_DIR, filename))

## Part 11 — Wrong test predictions (per class)

For every class, only the test images the model got WRONG, grouped per class (instead of one mixed grid) and styled red throughout.

In [ ]:


WRONG_SAMPLES_PER_CLASS = 4
wrong_rng = np.random.RandomState(SEED)

def save_wrong_prediction_grid(class_name, class_idx, save_path,
                                max_display=WRONG_SAMPLES_PER_CLASS):
    candidates = [
        i for i in range(len(test_paths))
        if y_true[i] == class_idx and y_pred[i] != class_idx
    ]
    if not candidates:
        print(f"  {class_name:30s} -> 0 wrong predictions, skipped")
        return

    n_sample = min(max_display, len(candidates))
    sample = list(wrong_rng.choice(candidates, size=n_sample, replace=False))

    fig, axes = plt.subplots(1, len(sample), figsize=(len(sample) * 3.5, 4.9))
    fig.patch.set_facecolor("white")
    if len(sample) == 1:
        axes = [axes]

    for ax, idx in zip(axes, sample):
        img = load_image_for_display(test_paths[idx])
        predicted_label = CLASS_NAMES[y_pred[idx]]
        confidence = y_conf[idx]

        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(COLORS["wrong"])
            spine.set_linewidth(3)

        track_ax = ax.inset_axes([0.0, -0.20, 1.0, 0.05])
        track_ax.barh([0], [1], color=COLORS["panel"], height=1)
        track_ax.barh([0], [confidence], color=COLORS["wrong"], height=1)
        track_ax.set_xlim(0, 1)
        track_ax.axis("off")

        ax.set_xlabel(
            f"Predicted: {predicted_label.replace('_', ' ')}\n{confidence*100:.1f}% confidence",
            fontsize=9, color=COLORS["wrong"], fontweight="bold", labelpad=22,
        )

    fig.suptitle(f"Wrong predictions \u2014 actual: {class_name.replace('_', ' ')}",
                 fontsize=14, fontweight="bold", color=COLORS["text"], y=1.05)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

wrong_total = int((y_true != y_pred).sum())
print(f"Total wrong test predictions: {wrong_total} / {len(y_true)}\n")
print("Generating WRONG per-class prediction grids...\n")
for class_idx, class_name in enumerate(CLASS_NAMES):
    filename = f"wrong_predictions_{class_name}.png"
    save_wrong_prediction_grid(class_name, class_idx, os.path.join(OUTPUT_DIR, filename))

## Part 12 — Export to TFLite

Converts the trained EfficientNetB0 model directly to standard (FP32) and INT8-quantized `.tflite` models (no `.keras` export).

In [ ]:


# --- Standard TFLite conversion (FP32 — recommended for Render backend) ---
converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()
TFLITE_PATH = os.path.join(OUTPUT_DIR, "waste_classifier.tflite")
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_model)
print(f"Saved standard TFLite model: {TFLITE_PATH}")

# --- INT8 quantized TFLite conversion (smaller/faster option) ---
def representative_dataset():
    # train_ds_raw yields (image, label, sample_weight) — unpack all three
    for batch in train_ds_raw.take(100):
        images = batch[0]          # first element is always the image tensor
        for image in images:
            yield [tf.expand_dims(image, 0)]

converter_int8 = tf.lite.TFLiteConverter.from_keras_model(model)
converter_int8.optimizations = [tf.lite.Optimize.DEFAULT]
converter_int8.representative_dataset = representative_dataset

tflite_int8_model = converter_int8.convert()
TFLITE_INT8_PATH = os.path.join(OUTPUT_DIR, "waste_classifier_int8.tflite")
with open(TFLITE_INT8_PATH, "wb") as f:
    f.write(tflite_int8_model)
print(f"Saved INT8 quantized TFLite model: {TFLITE_INT8_PATH}")

tflite_size_mb = os.path.getsize(TFLITE_PATH) / (1024 * 1024)
tflite_int8_size_mb = os.path.getsize(TFLITE_INT8_PATH) / (1024 * 1024)
print(f"\nTFLite size (standard): {tflite_size_mb:.2f} MB")
print(f"TFLite size (INT8):     {tflite_int8_size_mb:.2f} MB")

## Part 13 — Class label mapping

Maps class index to class name for the Android app, saved as `class_mapping.json`.

In [ ]:


class_mapping = {str(idx): name for idx, name in enumerate(CLASS_NAMES)}
CLASS_MAPPING_PATH = os.path.join(OUTPUT_DIR, "class_mapping.json")
with open(CLASS_MAPPING_PATH, "w") as f:
    json.dump(class_mapping, f, indent=2)

print(f"Saved: {CLASS_MAPPING_PATH}")
print(json.dumps(class_mapping, indent=2))

## Part 14 — Results summary and dashboard

A text summary (with a target-met check) plus a one-page performance dashboard image.

In [ ]:


total_images = sum(class_counts.values())
summary_lines = [
    "WISE SYSTEM - WASTE CLASSIFICATION TRAINING SUMMARY",
    "=" * 50,
    f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
    "Backbone:              EfficientNetB0",
    "",
    f"Total classes:         {NUM_CLASSES}",
    f"Total images:          {total_images}",
    f"Train images:          {len(train_paths)}",
    f"Validation images:     {len(val_paths)}",
    f"Test images:           {len(test_paths)}",
    "",
    f"Final test accuracy:   {test_accuracy:.4f}",
    f"Final test precision:  {test_precision:.4f}",
    f"Final test recall:     {test_recall:.4f}",
    f"Final test F1 score:   {test_f1:.4f}",
    f"Target accuracy:       {TARGET_ACCURACY:.4f}",
    f"Target met:            {'YES' if test_accuracy >= TARGET_ACCURACY else 'NO'}",
    "",
    f"TFLite size (std):     {tflite_size_mb:.2f} MB",
    f"TFLite size (INT8):    {tflite_int8_size_mb:.2f} MB",
]
summary_text = "\n".join(summary_lines)

SUMMARY_PATH = os.path.join(OUTPUT_DIR, "training_summary.txt")
with open(SUMMARY_PATH, "w") as f:
    f.write(summary_text)

print(summary_text)
print(f"\nSaved: {SUMMARY_PATH}")

### Performance dashboard

In [ ]:


def save_dashboard(save_path):
    fig = plt.figure(figsize=(13, 7.5))
    fig.patch.set_facecolor("white")
    gs = gridspec.GridSpec(3, 4, figure=fig, hspace=0.75, wspace=0.45,
                            top=0.85, bottom=0.06, left=0.055, right=0.97)

    fig.suptitle("WISE System \u2014 Waste Classification Dashboard",
                 fontsize=19, fontweight="bold", color=COLORS["text"],
                 x=0.055, ha="left", y=0.98)
    fig.text(0.055, 0.925,
             f"EfficientNetB0 \u00b7 {NUM_CLASSES} classes \u00b7 {total_images} images \u00b7 evaluated on the held-out test set",
             fontsize=10.5, color=COLORS["subtext"])

    metrics = [
        ("Accuracy", test_accuracy, COLORS["accuracy"]),
        ("Precision", test_precision, COLORS["precision"]),
        ("Recall", test_recall, COLORS["recall"]),
        ("F1 score", test_f1, COLORS["loss"]),
    ]

    for i, (label, value, color) in enumerate(metrics):
        ax = fig.add_subplot(gs[0, i])
        ax.axis("off")
        ax.add_patch(FancyBboxPatch((0.03, 0.06), 0.94, 0.88,
                                     boxstyle="round,pad=0.0,rounding_size=0.08",
                                     linewidth=0, facecolor=COLORS["panel"],
                                     transform=ax.transAxes, zorder=1))
        ax.add_patch(FancyBboxPatch((0.03, 0.85), 0.94, 0.09,
                                     boxstyle="round,pad=0.0,rounding_size=0.04",
                                     linewidth=0, facecolor=color,
                                     transform=ax.transAxes, zorder=2))
        ax.text(0.5, 0.48, f"{value*100:.1f}%", ha="center", va="center",
                fontsize=24, fontweight="bold", color=color,
                transform=ax.transAxes, zorder=3)
        ax.text(0.5, 0.19, label, ha="center", va="center",
                fontsize=11.5, color=COLORS["subtext"],
                transform=ax.transAxes, zorder=3)

    info_rows = [
        ("Total classes", str(NUM_CLASSES)),
        ("Total images", str(total_images)),
        ("TFLite (FP32) size", f"{tflite_size_mb:.2f} MB"),
        ("TFLite (INT8) size", f"{tflite_int8_size_mb:.2f} MB"),
    ]
    ax_info = fig.add_subplot(gs[1:, :2])
    ax_info.axis("off")
    ax_info.set_title("Model overview", fontsize=13.5, fontweight="bold",
                       color=COLORS["text"], loc="left", pad=10)
    for i, (label, value) in enumerate(info_rows):
        y = 0.76 - i * 0.23
        ax_info.text(0.0, y, label, fontsize=11.5, color=COLORS["subtext"],
                     transform=ax_info.transAxes)
        ax_info.text(1.0, y, value, fontsize=12.5, fontweight="bold", ha="right",
                     color=COLORS["text"], transform=ax_info.transAxes)
        ax_info.axhline(y - 0.09, color=COLORS["grid"], linewidth=1,
                         xmin=0.0, xmax=1.0)

    ax_bar = fig.add_subplot(gs[1:, 2:])
    bar_labels = [m[0] for m in metrics]
    bar_values = [m[1] for m in metrics]
    bar_colors = [m[2] for m in metrics]
    y_pos = np.arange(len(bar_labels))

    ax_bar.barh(y_pos, bar_values, color=bar_colors, height=0.55, zorder=3)
    ax_bar.set_yticks(y_pos)
    ax_bar.set_yticklabels(bar_labels)
    ax_bar.invert_yaxis()
    ax_bar.axvline(TARGET_ACCURACY, color=COLORS["subtext"], linestyle="--",
                   linewidth=1.2, zorder=2)
    ax_bar.text(TARGET_ACCURACY + 0.015, -0.6, f"{TARGET_ACCURACY*100:.0f}% target",
                fontsize=9, color=COLORS["subtext"], va="bottom")
    ax_bar.set_xlim(0, 1)
    ax_bar.set_title("Metric comparison", fontsize=13.5, fontweight="bold",
                      color=COLORS["text"], loc="left", pad=10)
    for i, v in enumerate(bar_values):
        ax_bar.text(v + 0.02, i, f"{v*100:.1f}%", va="center", fontsize=10,
                    fontweight="bold", color=COLORS["text"])
    ax_bar.grid(axis="x", alpha=0.6, linestyle="-", color=COLORS["grid"], zorder=0)
    for spine in ("top", "right", "left"):
        ax_bar.spines[spine].set_visible(False)
    ax_bar.tick_params(left=False)

    plt.savefig(save_path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.show()
    print(f"Saved: {save_path}")

save_dashboard(os.path.join(OUTPUT_DIR, "ai_model_dashboard.png"))

## Part 15 — Zip outputs

Bundles every generated file (no raw dataset images) into one ZIP.

In [ ]:


ZIP_PATH = "/content/wise_system_waste_outputs.zip"

with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for filename in sorted(os.listdir(OUTPUT_DIR)):
        filepath = os.path.join(OUTPUT_DIR, filename)
        if os.path.isfile(filepath):
            zf.write(filepath, arcname=filename)

zip_size_mb = os.path.getsize(ZIP_PATH) / (1024 * 1024)
print(f"Created ZIP: {ZIP_PATH} ({zip_size_mb:.2f} MB)")
print("\nContents:")
with zipfile.ZipFile(ZIP_PATH, "r") as zf:
    for name in zf.namelist():
        print(f"  - {name}")

## Part 16 — Download (TFLite only)

Downloads the `.tflite` models, `class_mapping.json`, and the reports ZIP for WISE deployment.

In [ ]:
from google.colab import files

files.download(TFLITE_PATH)
files.download(TFLITE_INT8_PATH)
files.download(CLASS_MAPPING_PATH)
files.download(ZIP_PATH)